# Figure 3: F1, C1 and C2

Seed 42. Six trained models from the chat-based run; training-only batch-normalisation recalibration along the parameter path.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import copy, hashlib, json, math, platform, random, sys, time, warnings
from dataclasses import dataclass, asdict, replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import torchvision
from IPython.display import display

torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(4)

from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

NETWORKS = ("F1", "C1", "C2")
SEEDS = (42,)
RUN_DIR = Path(os.environ.get("PAPER2_RUN_DIR", Path.cwd())).expanduser().resolve()
SOURCE_NOTEBOOK = RUN_DIR / "on_large_batch_training_chat_seed42_v2.ipynb"
data_candidates = [parent / "data" for parent in (RUN_DIR, *RUN_DIR.parents)]
DATA_ROOT = Path(os.environ.get("ASSIGNMENT2_DATA_DIR", next((path for path in data_candidates if path.is_dir()), RUN_DIR / "data")))
OUTPUT_ROOT = RUN_DIR / "results"
DOWNLOAD_DATA = True
RUN_TRAINING = True
RUN_PATHS = True
MAX_EPOCHS = 300
ALLOW_CAPPED_ANALYSIS = True
print(f"Device: {DEVICE}; seed: 42; new results: {OUTPUT_ROOT}")

@dataclass(frozen=True)
class Protocol:
    learning_rate: float = 1e-3
    adam_betas: tuple = (0.9, 0.999)
    adam_eps: float = 1e-8
    weight_decay: float = 0.0
    small_batch: int = 256
    large_batch_fraction: float = 0.10
    pixel_scale: str = "unit"
    bn_eps: float = 1e-5
    bn_momentum: float = 0.1
    output_batch_norm: bool = False
    dropout: float = 0.5
    min_epochs: int = 100
    patience: int = 20
    min_delta: float = 1e-4
    eval_batch: int = 256
    path_points: int = 61
    path_selector_seed: int = 20261022

CFG = Protocol()

source_notebook = json.loads(SOURCE_NOTEBOOK.read_text())
source_cells = ["".join(cell["source"]) for cell in source_notebook["cells"] if cell["cell_type"] == "code"]
IMPLEMENTATION_SHA256 = hashlib.sha256(json.dumps(source_cells, ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()

PAPER_SHA256 = "fbb3d2d981314adc7e5e3df7ec2c067f03d769a2a418c4d2580fa456cc5548d6"

ARCHITECTURE_NOTES = "Xavier; SAME C1 conv/pool; C2 13 conv; hidden affine-BN-ReLU; dense dropout; log-BN-variance path"

ENVIRONMENT = {
    "python": platform.python_version(), "torch": torch.__version__,
    "torchvision": torchvision.__version__, "numpy": np.__version__,
    "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
    "device_type": DEVICE.type, "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "device_name": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else platform.machine(),
    "num_workers": 0, "deterministic_algorithms": True,
    "torch_num_threads": torch.get_num_threads(),
    "torch_num_interop_threads": torch.get_num_interop_threads(),
    "platform": platform.platform(), "default_dtype": str(torch.get_default_dtype()),
}
CODE_REVISION = "gpt-chat-seed42-v2"
def stable_json(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), allow_nan=False)

def digest(obj):
    return hashlib.sha256(stable_json(obj).encode()).hexdigest()

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def json_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False))
    os.replace(tmp, path)

def torch_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(value, tmp)
    os.replace(tmp, path)

def load_checkpoint(path):
    # Only load checkpoints created by this notebook: they contain Python RNG state.
    return torch.load(path, map_location="cpu", weights_only=False)

def make_loader(dataset, batch_size, shuffle, seed, drop_last=False):
    if batch_size < 2 and shuffle:
        raise ValueError("Training batch must be at least 2 for dense batch normalization")
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=drop_last, num_workers=0, generator=generator,
                      collate_fn=keep_cached_batch if isinstance(dataset, CachedTensorDataset) else None)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    parameter = next(model.parameters())
    total_loss, correct, count = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device=parameter.device, dtype=parameter.dtype)
        y = y.to(parameter.device)
        logits = model(x)
        total_loss += float(F.cross_entropy(logits, y, reduction="sum").item())
        correct += int((logits.argmax(1) == y).sum().item())
        count += len(y)
    if count != len(loader.dataset):
        raise RuntimeError("Evaluation must cover the full dataset, including its last batch")
    return {"loss": total_loss / count, "accuracy": 100.0 * correct / count, "n": count}

def state_hash(state):
    h = hashlib.sha256()
    for name, value in state.items():
        h.update(name.encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

In [ ]:
class CachedTensorDataset(Dataset):
    def __init__(self, images, targets):
        self.images = images
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        return self.images[index], self.targets[index]

    def __getitems__(self, indices):
        return self.images[indices], self.targets[indices]


def keep_cached_batch(batch):
    return batch


def cache_dataset(dataset, cfg):
    raw_images = torch.as_tensor(dataset.data)
    if raw_images.ndim == 3:
        raw_images = raw_images.unsqueeze(1)
    else:
        raw_images = raw_images.permute(0, 3, 1, 2).contiguous()
    images = raw_images.to(dtype=torch.float32).div_(255)
    if cfg.pixel_scale == "raw255":
        images.mul_(255)
    targets = torch.as_tensor(dataset.targets, dtype=torch.long)
    for index in (0, 1, 42, len(dataset) - 1):
        reference_image, reference_target = dataset[index]
        if not torch.equal(images[index], reference_image) or targets[index].item() != reference_target:
            raise RuntimeError("Cached data differs from the original torchvision transform")
    return CachedTensorDataset(images.to(DEVICE), targets.to(DEVICE))

class PixelTransform:
    def __init__(self, scale):
        if scale not in ("unit", "raw255"):
            raise ValueError("pixel_scale must be unit or raw255")
        self.scale = scale
        self.to_tensor = transforms.ToTensor()
    def __call__(self, image):
        value = self.to_tensor(image)
        return value if self.scale == "unit" else value * 255.0

_DATA_CACHE = {}

def get_data(network, cfg=CFG):
    dataset_name = "MNIST" if network == "F1" else "CIFAR10"
    key = (dataset_name, cfg.pixel_scale, str(DATA_ROOT.resolve()))
    if key not in _DATA_CACHE:
        cls = datasets.MNIST if dataset_name == "MNIST" else datasets.CIFAR10
        transform = PixelTransform(cfg.pixel_scale)
        train = cls(str(DATA_ROOT), train=True, download=DOWNLOAD_DATA, transform=transform)
        test = cls(str(DATA_ROOT), train=False, download=DOWNLOAD_DATA, transform=transform)
        expected_train = 60000 if dataset_name == "MNIST" else 50000
        assert len(train) == expected_train and len(test) == 10000
        hashes = {}
        for split, ds in (("train", train), ("test", test)):
            h = hashlib.sha256()
            data = ds.data.numpy() if torch.is_tensor(ds.data) else np.asarray(ds.data)
            h.update(memoryview(np.ascontiguousarray(data)).cast("B"))
            h.update(np.asarray(ds.targets, dtype=np.int64).tobytes())
            hashes[split] = h.hexdigest()
        _DATA_CACHE[key] = (cache_dataset(train, cfg), cache_dataset(test, cfg), {"dataset": dataset_name, "hashes": hashes,
                                       "train_n": len(train), "test_n": len(test)})
    return _DATA_CACHE[key]

In [ ]:
class SameConv2d(nn.Conv2d):
    def forward(self, x):
        h, w = x.shape[-2:]
        sh, sw = self.stride
        kh, kw = self.kernel_size
        dh, dw = self.dilation
        ph = max((math.ceil(h/sh)-1)*sh + dh*(kh-1)+1-h, 0)
        pw = max((math.ceil(w/sw)-1)*sw + dw*(kw-1)+1-w, 0)
        x = F.pad(x, (pw//2, pw-pw//2, ph//2, ph-ph//2))
        return F.conv2d(x, self.weight, self.bias, self.stride, 0, self.dilation, self.groups)

class SameMaxPool2d(nn.Module):
    def __init__(self, kernel_size=3, stride=2):
        super().__init__()
        self.kernel_size, self.stride = kernel_size, stride
    def forward(self, x):
        h, w = x.shape[-2:]
        k, s = self.kernel_size, self.stride
        ph, pw = max((math.ceil(h/s)-1)*s+k-h, 0), max((math.ceil(w/s)-1)*s+k-w, 0)
        x = F.pad(x, (pw//2, pw-pw//2, ph//2, ph-ph//2), value=-float("inf"))
        return F.max_pool2d(x, k, s)

def conv_block(cin, cout, kernel, stride, cfg):
    return [SameConv2d(cin, cout, kernel, stride=stride, bias=True),
            nn.BatchNorm2d(cout, eps=cfg.bn_eps, momentum=cfg.bn_momentum), nn.ReLU()]

def dense_block(nin, nout, cfg, dropout=False):
    layers = [nn.Linear(nin, nout, bias=True),
              nn.BatchNorm1d(nout, eps=cfg.bn_eps, momentum=cfg.bn_momentum), nn.ReLU()]
    return layers + ([nn.Dropout(cfg.dropout)] if dropout else [])

def initialize(module):
    if isinstance(module, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)
        module.reset_running_stats()

def make_model(network, cfg=CFG):
    if network == "F1":
        layers = [nn.Flatten()]
        previous = 784
        for _ in range(5):
            layers += dense_block(previous, 512, cfg)
            previous = 512
        layers += [nn.Linear(512, 10)]
    elif network == "C1":
        layers = conv_block(3, 64, 5, 2, cfg) + [SameMaxPool2d(3, 2)]
        layers += conv_block(64, 64, 5, 2, cfg) + [SameMaxPool2d(3, 2), nn.Flatten()]
        layers += dense_block(64*2*2, 384, cfg, True)
        layers += dense_block(384, 192, cfg, True) + [nn.Linear(192, 10)]
    elif network == "C2":
        layers, previous = [], 3
        for channels, depth in ((64,2), (128,2), (256,3), (512,3), (512,3)):
            for _ in range(depth):
                layers += conv_block(previous, channels, 3, 1, cfg)
                previous = channels
            layers += [nn.MaxPool2d(2, stride=2)]
        layers += [nn.Flatten()] + dense_block(512, 512, cfg, True) + [nn.Linear(512, 10)]
    else:
        raise ValueError(network)
    if cfg.output_batch_norm:
        layers += [nn.BatchNorm1d(10, eps=cfg.bn_eps, momentum=cfg.bn_momentum)]
    model = nn.Sequential(*layers)
    model.apply(initialize)
    return model

def architecture_checks(cfg=CFG):
    rows = []
    with torch.random.fork_rng(devices=[]):
        for network in ("F1", "C1", "C2"):
            seed_all(123)
            model = make_model(network, cfg).eval()
            shape = (2, 1, 28, 28) if network == "F1" else (2, 3, 32, 32)
            with torch.no_grad():
                output = model(torch.zeros(shape))
            assert output.shape == (2, 10) and torch.isfinite(output).all()
            rows.append({"network": network,
                         "parameters": sum(p.numel() for p in model.parameters()),
                         "conv_layers": sum(isinstance(m, nn.Conv2d) for m in model.modules()),
                         "linear_layers": sum(isinstance(m, nn.Linear) for m in model.modules()),
                         "dropout_layers": sum(isinstance(m, nn.Dropout) for m in model.modules())})
            del model
    return pd.DataFrame(rows)

In [ ]:
def training_settings(cfg):
    analysis_fields = {"path_points", "path_selector_seed"}
    return {key: value for key, value in asdict(cfg).items() if key not in analysis_fields}

def capture_rng(loaders):
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "loaders": {key: loader.generator.get_state() for key, loader in loaders.items()}}

def restore_rng(state, loaders):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None:
        if not torch.cuda.is_available():
            raise RuntimeError("A CUDA checkpoint cannot exactly resume on this CPU runtime")
        torch.cuda.set_rng_state_all(state["cuda"])
    for key, loader in loaders.items():
        loader.generator.set_state(state["loaders"][key])

def experiment_manifest(network, regime, seed, batch_size, cfg, data_info):
    return {"schema": 1, "implementation_sha256": IMPLEMENTATION_SHA256,
            "code_revision": CODE_REVISION,
            "paper_sha256": PAPER_SHA256, "architecture_notes": ARCHITECTURE_NOTES,
            "network": network, "regime": regime, "seed": seed, "batch_size": batch_size,
            "protocol": training_settings(cfg), "data": data_info, "environment": ENVIRONMENT}

def experiment_location(network, regime, seed, cfg, data_info, root=OUTPUT_ROOT):
    batch_size = cfg.small_batch if regime == "SB" else int(data_info["train_n"] * cfg.large_batch_fraction)
    manifest = experiment_manifest(network, regime, seed, batch_size, cfg, data_info)
    directory = Path(root) / f"{network}_{regime}_seed{seed}_{digest(manifest)[:12]}"
    return directory, manifest

def write_trial_summaries(directory, checkpoint):
    # The atomic checkpoint is authoritative if interruption occurs during summary writes.
    history = checkpoint["history"]
    last = history[-1]
    manifest = checkpoint["manifest"]
    json_write(directory / "manifest.json", manifest)
    json_write(directory / "history.json", history)
    json_write(directory / "result.json", {
        "network": manifest["network"], "regime": manifest["regime"], "seed": manifest["seed"],
        "status": checkpoint["status"], "completed_epoch": checkpoint["completed_epoch"],
        "global_step": checkpoint["global_step"],
        "initial_state_sha256": checkpoint["initial_state_sha256"],
        "train": {"loss": last["train_loss"], "accuracy": last["train_accuracy"], "n": manifest["data"]["train_n"]},
        "test": {"loss": last["test_loss"], "accuracy": last["test_accuracy"], "n": manifest["data"]["test_n"]},
        "checkpoint_model_sha256": state_hash(checkpoint["model"]),
    })

def train_trial(network, regime, seed, train_set, test_set, data_info,
                cfg=CFG, max_epochs=MAX_EPOCHS, root=OUTPUT_ROOT, device=DEVICE):
    if regime not in ("SB", "LB"):
        raise ValueError(regime)
    if cfg.min_epochs < 1 or cfg.patience < 1 or cfg.min_delta < 0:
        raise ValueError("Invalid stopping-rule settings")
    if max_epochs is not None and max_epochs < 1:
        raise ValueError("MAX_EPOCHS must be positive or None")
    directory, manifest = experiment_location(network, regime, seed, cfg, data_info, root)
    batch_size = manifest["batch_size"]
    if batch_size > len(train_set):
        raise ValueError("Batch exceeds training dataset")
    directory.mkdir(parents=True, exist_ok=True)
    checkpoint_path = directory / "last.pt"
    seed_all(seed)
    model = make_model(network, cfg).to(device)
    initial_hash = state_hash(model.state_dict())
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate,
                                 betas=cfg.adam_betas, eps=cfg.adam_eps,
                                 weight_decay=cfg.weight_decay, amsgrad=False,
                                 foreach=False, fused=False)
    loaders = {
        "train": make_loader(train_set, batch_size, True, seed + 1000, True),
        "eval_train": make_loader(train_set, cfg.eval_batch, False, seed + 2000),
        "eval_test": make_loader(test_set, cfg.eval_batch, False, seed + 3000),
    }
    completed_epoch, global_step, history = 0, 0, []
    best_reference, stale_epochs, status = float("inf"), 0, "running"
    if checkpoint_path.exists():
        saved = load_checkpoint(checkpoint_path)
        if stable_json(saved["manifest"]) != stable_json(manifest):
            raise RuntimeError("Checkpoint manifest mismatch; refusing to overwrite")
        if saved["initial_state_sha256"] != initial_hash:
            raise RuntimeError("Initialization changed; use a new result directory")
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        completed_epoch, global_step = saved["completed_epoch"], saved["global_step"]
        history, best_reference = saved["history"], saved["best_reference"]
        stale_epochs, status = saved["stale_epochs"], saved["status"]
        restore_rng(saved["rng"], loaders)
        write_trial_summaries(directory, saved)
        if status == "converged" or (max_epochs is not None and completed_epoch >= max_epochs):
            print(f"{network} {regime} seed={seed}: reusing {completed_epoch} saved epochs")
            return directory
    json_write(directory / "manifest.json", manifest)
    while max_epochs is None or completed_epoch < max_epochs:
        model.train()
        seen, stochastic_loss = 0, 0.0
        started = time.perf_counter()
        for x, y in loaders["train"]:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x), y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite training loss in {directory.name}; last checkpoint preserved")
            loss.backward()
            optimizer.step()
            global_step += 1
            stochastic_loss += float(loss.item()) * len(y)
            seen += len(y)
        train_metrics = evaluate(model, loaders["eval_train"])
        test_metrics = evaluate(model, loaders["eval_test"])
        if not math.isfinite(train_metrics["loss"]) or not math.isfinite(test_metrics["loss"]):
            raise FloatingPointError("Nonfinite evaluation loss; refusing to checkpoint it")
        completed_epoch += 1
        if train_metrics["loss"] < best_reference - cfg.min_delta:
            best_reference, stale_epochs = train_metrics["loss"], 0
        else:
            stale_epochs += 1
        converged = completed_epoch >= cfg.min_epochs and stale_epochs >= cfg.patience
        capped = max_epochs is not None and completed_epoch >= max_epochs
        status = "converged" if converged else ("budget_exhausted" if capped else "running")
        history.append({"epoch": completed_epoch, "global_step": global_step,
                        "training_examples_seen": seen,
                        "stochastic_training_loss": stochastic_loss / seen,
                        "train_loss": train_metrics["loss"], "test_loss": test_metrics["loss"],
                        "train_accuracy": train_metrics["accuracy"], "test_accuracy": test_metrics["accuracy"],
                        "stale_epochs": stale_epochs, "seconds": time.perf_counter()-started})
        payload = {"manifest": manifest, "initial_state_sha256": initial_hash,
                   "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                   "scheduler": None, "completed_epoch": completed_epoch,
                   "global_step": global_step, "history": history,
                   "best_reference": best_reference, "stale_epochs": stale_epochs,
                   "status": status, "max_epochs_at_save": max_epochs,
                   "rng": capture_rng(loaders)}
        torch_write(checkpoint_path, payload)
        write_trial_summaries(directory, payload)
        print(f"{network} {regime} seed={seed} epoch={completed_epoch} "
              f"train_loss={train_metrics['loss']:.6f} test_acc={test_metrics['accuracy']:.2f}% "
              f"seconds={history[-1]['seconds']:.2f}", flush=True)
        if converged:
            break
    return directory

def require_usable(checkpoint, allow_capped=False):
    if checkpoint["status"] != "converged" and not allow_capped:
        raise RuntimeError("Run is not converged. Increase MAX_EPOCHS and resume; "
                           "or explicitly enable ALLOW_CAPPED_ANALYSIS for provisional analysis.")

def saved_trial(network, regime, seed, cfg=CFG):
    train, test, data_info = get_data(network, cfg)
    directory, manifest = experiment_location(network, regime, seed, cfg, data_info, OUTPUT_ROOT)
    path = directory / "last.pt"
    if not path.exists():
        raise FileNotFoundError(f"Train this trial first: {path}")
    saved = load_checkpoint(path)
    if stable_json(saved["manifest"]) != stable_json(manifest):
        raise RuntimeError("Saved experiment does not match current configuration")
    require_usable(saved, ALLOW_CAPPED_ANALYSIS)
    if saved["status"] == "budget_exhausted":
        print(f"{network} {regime}: budget exhausted; path is labelled provisional.", flush=True)
    return saved, directory, train, test

In [ ]:
ANALYSIS_NOTEBOOK = RUN_DIR / "on_large_batch_training_chat_seed42_v2_bn_recalibrated.ipynb"
ANALYSIS_ROOT = RUN_DIR / "results_bn_recalibrated"
CALIBRATION_BATCH = 512
analysis_notebook = json.loads(ANALYSIS_NOTEBOOK.read_text())
for cell_index in (1, 2, 3, 4, 7):
    if "".join(analysis_notebook["cells"][cell_index]["source"]) != "".join(source_notebook["cells"][cell_index]["source"]):
        raise RuntimeError("Training code differs from the archived source that produced the checkpoints")
analysis_sources = ["".join(cell["source"]) for cell in analysis_notebook["cells"] if cell["cell_type"] == "code"]
ANALYSIS_SHA256 = hashlib.sha256(json.dumps(analysis_sources, ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()


@torch.no_grad()
def recalibrate_batchnorm(model, training_images):
    model.eval()
    normalizations = [module for module in model.modules()
                      if isinstance(module, nn.modules.batchnorm._BatchNorm)]
    momenta = [module.momentum for module in normalizations]
    for module in normalizations:
        module.reset_running_stats()
        module.train()
    count = 0
    try:
        for start in range(0, len(training_images), CALIBRATION_BATCH):
            images = training_images[start:start + CALIBRATION_BATCH]
            for module in normalizations:
                module.momentum = len(images) / (count + len(images))
            model(images)
            count += len(images)
    finally:
        for module, momentum in zip(normalizations, momenta):
            module.momentum = momentum
        model.eval()


def parameter_path(network, seed, cfg=CFG):
    small, _, train, test = saved_trial(network, "SB", seed, cfg)
    large, _, _, _ = saved_trial(network, "LB", seed, cfg)
    assert small["initial_state_sha256"] == large["initial_state_sha256"]
    metadata = {
        "network": network, "seed": seed, "protocol": asdict(cfg),
        "training_implementation_sha256": IMPLEMENTATION_SHA256,
        "analysis_implementation_sha256": ANALYSIS_SHA256,
        "sb_model_sha256": state_hash(small["model"]),
        "lb_model_sha256": state_hash(large["model"]),
        "sb_manifest": small["manifest"], "lb_manifest": large["manifest"],
        "statuses": [small["status"], large["status"]],
        "bn_buffers": "training-only sample-weighted recalibration at every alpha, including endpoints",
        "calibration_batch": CALIBRATION_BATCH,
        "dropout_during_calibration": False,
        "training_performed_for_this_analysis": False,
        "evaluation_dtype": "float32",
        "provisional": any(saved["status"] != "converged" for saved in (small, large)),
        "stopping_label_caveat": "Converged denotes the recorded patience rule, not a mathematical convergence guarantee.",
    }
    folder = ANALYSIS_ROOT / "analysis" / f"path_{network}_seed{seed}_{digest(metadata)[:12]}"
    folder.mkdir(parents=True, exist_ok=True)
    csv_path = folder / "path.csv"
    alphas = np.unique(np.round(np.r_[np.linspace(-1., 2., cfg.path_points), 0., 1.], 8))
    if csv_path.exists():
        if stable_json(json.loads((folder / "manifest.json").read_text())) != stable_json(metadata):
            raise RuntimeError("Analysis cache metadata mismatch")
        frame = pd.read_csv(csv_path)
        assert np.array_equal(frame.alpha.to_numpy(), alphas)
        assert np.isfinite(frame.to_numpy()).all()
        return frame, folder
    json_write(folder / "manifest.json", metadata)
    model = make_model(network, cfg).to(DEVICE).eval()
    train_loader = make_loader(train, cfg.eval_batch, False, seed + 4000)
    test_loader = make_loader(test, cfg.eval_batch, False, seed + 5000)
    rows = []
    for alpha in alphas:
        model.load_state_dict(small["model"])
        with torch.no_grad():
            for name, parameter in model.named_parameters():
                interpolated = (1 - alpha) * small["model"][name].double() + alpha * large["model"][name].double()
                parameter.copy_(interpolated)
        parameters_before = state_hash(dict(model.named_parameters()))
        recalibrate_batchnorm(model, train.images)
        assert state_hash(dict(model.named_parameters())) == parameters_before
        train_metrics, test_metrics = evaluate(model, train_loader), evaluate(model, test_loader)
        if not math.isfinite(train_metrics["loss"]) or not math.isfinite(test_metrics["loss"]):
            raise FloatingPointError(f"Nonfinite recalibrated path at {network}, alpha={alpha}")
        rows.append({
            "alpha": float(alpha), "train_loss": train_metrics["loss"],
            "test_loss": test_metrics["loss"], "train_accuracy": train_metrics["accuracy"],
            "test_accuracy": test_metrics["accuracy"],
        })
        print(f"{network} alpha={alpha:.2f}: train CE={train_metrics['loss']:.5f}, test accuracy={test_metrics['accuracy']:.2f}%", flush=True)
    frame = pd.DataFrame(rows)
    for alpha, saved in ((0., small), (1., large)):
        model.load_state_dict(saved["model"])
        recalibrate_batchnorm(model, train.images)
        independent = {"train": evaluate(model, train_loader), "test": evaluate(model, test_loader)}
        endpoint = frame.loc[frame.alpha == alpha].iloc[0]
        for split in ("train", "test"):
            assert np.isclose(endpoint[f"{split}_loss"], independent[split]["loss"], rtol=1e-6, atol=1e-7)
            assert np.isclose(endpoint[f"{split}_accuracy"], independent[split]["accuracy"], rtol=0, atol=1e-8)
    temporary = csv_path.with_suffix(".tmp.csv")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, csv_path)
    return frame, folder


def path_trial_seed(network, seeds=SEEDS, cfg=CFG):
    rng = np.random.default_rng(cfg.path_selector_seed + ("F1","C1","C2").index(network))
    return int(rng.choice(seeds))

def plot_parameter_paths(frames):
    fig, axes = plt.subplots(1, len(frames), figsize=(5.5*len(frames),4), squeeze=False)
    for ax, (network, frame) in zip(axes.flat, frames.items()):
        right = ax.twinx()
        for split, style in (("train","-"),("test","--")):
            ax.plot(frame.alpha, frame[f"{split}_loss"], style, color="blue", label=split.title())
            right.plot(frame.alpha, frame[f"{split}_accuracy"], style, color="red")
        for x in (0,1): ax.axvline(x, color="0.6", lw=0.6)
        ax.set(title=network, xlabel="α (0 = SB; 1 = LB)", ylabel="Cross-entropy", xlim=(-1, 2))
        right.set(ylabel="Accuracy (%)", ylim=(0,100))
        ax.legend(); ax.tick_params(axis="y", colors="blue")
        right.tick_params(axis="y", colors="red")
    fig.tight_layout()
    return fig

In [ ]:
display(architecture_checks(CFG))

In [ ]:
if RUN_TRAINING:
    for network in NETWORKS:
        train_set, test_set, data_info = get_data(network, CFG)
        for seed in SEEDS:
            initial_hashes = []
            for regime in ("SB", "LB"):
                folder = train_trial(network, regime, seed, train_set, test_set, data_info,
                                     cfg=CFG, max_epochs=MAX_EPOCHS, root=OUTPUT_ROOT, device=DEVICE)
                initial_hashes.append(json.loads((folder / "result.json").read_text())["initial_state_sha256"])
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
            assert len(set(initial_hashes)) == 1, "SB/LB did not share initial weights"

In [ ]:
if RUN_PATHS:
    path_frames, path_folders = {}, {}
    for network in NETWORKS:
        chosen = path_trial_seed(network, SEEDS, CFG)
        path_frames[network], path_folders[network] = parameter_path(network, chosen, CFG)
    endpoint_rows = []
    for network, frame in path_frames.items():
        for endpoint, regime in ((0.0, "SB"), (1.0, "LB")):
            measured = frame.loc[frame.alpha == endpoint].iloc[0].to_dict()
            endpoint_rows.append({"network": network, "regime": regime, **measured})
    json_write(ANALYSIS_ROOT / "endpoint_metrics.json", endpoint_rows)
    display(pd.DataFrame(endpoint_rows))
    statuses = {network: json.loads((folder / "manifest.json").read_text())["statuses"]
                for network, folder in path_folders.items()}
    json_write(ANALYSIS_ROOT / "figure_status.json", {
        "seed": 42, "trials": 1, "statuses": statuses,
        "training_performed_for_correction": False,
        "bn_protocol": "training-only recalibration at every alpha, including endpoints",
        "analysis_implementation_sha256": ANALYSIS_SHA256,
        "checkpoint_stopping_rule_is_not_convergence_proof": True,
    })
    fig = plot_parameter_paths(path_frames)
    fig.suptitle("Figure 3: seed 42; training-only BN recalibration")
    fig.tight_layout()
    fig.savefig(ANALYSIS_ROOT / "figure3_chat_seed42_v2_bn_recalibrated.pdf")
    fig.savefig(ANALYSIS_ROOT / "figure3_chat_seed42_v2_bn_recalibrated.png", dpi=180)
    plt.show()
